In [1]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import boxcox
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.cluster import KMeans
from sklearn.cluster import AgglomerativeClustering, DBSCAN, Birch
from sklearn.metrics import silhouette_score
from scipy.spatial.distance import cdist

In [3]:
df = pd.read_csv("Fashion_Retail_Sales.csv")
df.head()

,Customer Reference ID,Item Purchased,Purchase Amount (USD),Date Purchase,Review Rating,Payment Method
0,4018,Handbag,4619.0,05-02-2023,NaN,Credit Card
1,4115,Tunic,2456.0,11-07-2023,2.0,Credit Card
2,4019,Tank Top,2102.0,23-03-2023,4.1,Cash
3,4097,Leggings,3126.0,15-03-2023,3.2,Cash
4,3997,Wallet,3003.0,27-11-2022,4.7,Cash


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3400 entries, 0 to 3399
Data columns (total 6 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Customer Reference ID  3400 non-null   int64  
 1   Item Purchased         3400 non-null   object 
 2   Purchase Amount (USD)  2750 non-null   float64
 3   Date Purchase          3400 non-null   object 
 4   Review Rating          3076 non-null   float64
 5   Payment Method         3400 non-null   object 
dtypes: float64(2), int64(1), object(3)
memory usage: 159.5+ KB


In [7]:
df.describe ()

,Customer Reference ID,Purchase Amount (USD),Review Rating
count,3400.000000,2750.000000,3076.000000
mean,4039.660588,156.709818,2.999057
std,48.122583,419.536669,1.156505
min,3957.000000,10.000000,1.000000
25%,3997.000000,57.000000,2.000000
50%,4040.000000,110.000000,3.000000
75%,4081.000000,155.750000,4.000000
max,4122.000000,4932.000000,5.000000


In [23]:
df.duplicated().sum()
# Om > 0, titta på dem innan du bestämmer dig för att droppa:
df[df.duplicated(keep=False)].sort_values('Customer Reference ID')

,Customer Reference ID,Item Purchased,Purchase Amount (USD),Date Purchase,Review Rating,Payment Method


In [25]:
df.groupby('Item Purchased')['Purchase Amount (USD)'].describe()[['min','max','mean']].sort_values('max', ascending=False)

,min,max,mean
Item Purchased,,,
Flip-Flops,18.0,4932.0,275.829268
Shorts,14.0,4872.0,171.648649
Sweater,26.0,4859.0,213.137255
Jeans,18.0,4771.0,272.250000
Tunic,13.0,4661.0,319.907407
Handbag,10.0,4619.0,214.711864
Romper,12.0,4465.0,197.104167
Bowtie,12.0,4418.0,197.840909
Gloves,10.0,4298.0,237.115385


In [27]:
# Hur många rader ligger över respektive under brytpunkten?
(df['Purchase Amount (USD)'] > 500).sum()
(df['Purchase Amount (USD)'] <= 500).sum()

# Finns ett mönster i VILKA rader som är "höga" - t.ex. kopplat till betalmetod eller datum?
df[df['Purchase Amount (USD)'] > 500].groupby('Payment Method').size()
df[df['Purchase Amount (USD)'] > 500]['Date Purchase'].describe()

count             43
unique            41
top       21-11-2022
freq               2
Name: Date Purchase, dtype: object

In [29]:
df[df['Purchase Amount (USD)'] > 500].groupby('Payment Method').size()

Payment Method
Cash           19
Credit Card    24
dtype: int64

In [33]:
df['Payment Method'].value_counts(normalize=True)

Payment Method
Credit Card    0.520588
Cash           0.479412
Name: proportion, dtype: float64

In [35]:
df['Customer Reference ID'].nunique()

166

In [37]:
df.groupby('Item Purchased')['Purchase Amount (USD)'].apply(lambda x: x.isna().mean()) 

Item Purchased
Backpack         0.225352
Belt             0.233333
Blazer           0.239437
Blouse           0.219178
Boots            0.200000
Bowtie           0.153846
Camisole         0.184211
Cardigan         0.152778
Coat             0.223881
Dress            0.228070
Flannel Shirt    0.158730
Flip-Flops       0.196078
Gloves           0.118644
Handbag          0.180556
Hat              0.176471
Hoodie           0.173333
Jacket           0.218750
Jeans            0.172414
Jumpsuit         0.192982
Kimono           0.164384
Leggings         0.209677
Loafers          0.210526
Onesie           0.169014
Overalls         0.114754
Pajamas          0.246914
Pants            0.174419
Polo Shirt       0.271186
Poncho           0.178082
Raincoat         0.202899
Romper           0.250000
Sandals          0.129630
Scarf            0.106061
Shorts           0.149425
Skirt            0.204545
Slippers         0.206897
Sneakers         0.169014
Socks            0.191781
Sun Hat          0.1600

In [39]:
customer_summary = df.groupby('Customer Reference ID').agg(
    frequency=('Item Purchased', 'count'),
    total_spend=('Purchase Amount (USD)', 'sum'),
    avg_spend=('Purchase Amount (USD)', 'mean'),
    avg_rating=('Review Rating', 'mean'),
    last_purchase=('Date Purchase', 'max')
).reset_index()

customer_summary.sort_values('total_spend', ascending=False).head(10)

,Customer Reference ID,frequency,total_spend,avg_spend,avg_rating,last_purchase
152,4109,16,9685.0,745.000000,2.416667,30-09-2023
83,4040,25,9657.0,536.500000,2.882609,31-07-2023
87,4044,24,8745.0,485.833333,3.373913,30-05-2023
118,4075,28,7416.0,296.640000,2.934615,31-07-2023
151,4108,26,6864.0,361.263158,2.844000,29-08-2023
110,4067,21,6528.0,326.400000,3.266667,30-06-2023
53,4010,24,6513.0,342.789474,3.031579,31-08-2023
146,4103,23,6375.0,318.750000,2.895455,31-03-2023
27,3984,19,6327.0,451.928571,3.205263,31-08-2023
45,4002,26,6320.0,287.272727,2.700000,31-01-2023


In [41]:
# Korrelation: spenderar man mer/oftare, men är man mindre nöjd?
customer_summary[['frequency', 'total_spend', 'avg_spend', 'avg_rating']].corr()

# Pareto-andelen på riktigt
total_revenue = customer_summary['total_spend'].sum()
top20_count = int(len(customer_summary) * 0.2)
top20_share = customer_summary.sort_values('total_spend', ascending=False).head(top20_count)['total_spend'].sum() / total_revenue
print(top20_share)

0.4421095620858007


In [43]:
customer_summary[['frequency', 'total_spend', 'avg_spend', 'avg_rating']].corr()

,frequency,total_spend,avg_spend,avg_rating
frequency,1.000000,0.332032,0.008708,-0.127619
total_spend,0.332032,1.000000,0.903682,0.050992
avg_spend,0.008708,0.903682,1.000000,0.096600
avg_rating,-0.127619,0.050992,0.096600,1.000000


In [49]:
#Conclusion
#An initial analysis indicated that total revenue per customer correlated most strongly with average purchase value (r=0.90). 
#However, a robustnesscheck revealed that this was driven almost exclusively by 43 transactions (1.3% data) involving unusually high amounts. 
#When these were excluded, purchase frequency emerged as the dominant driver (r=0.89). The focus can therefore be increasing visit frequency and 
#repeat purchases among the vast majority of customers, while the rare high-value purchases should be treated as a separate, smaller segment rather than a general trend.

In [47]:
df_clean = df[df['Purchase Amount (USD)'] <= 500]

customer_summary_clean = df_clean.groupby('Customer Reference ID').agg(
    frequency=('Item Purchased', 'count'),
    total_spend=('Purchase Amount (USD)', 'sum'),
    avg_spend=('Purchase Amount (USD)', 'mean'),
    avg_rating=('Review Rating', 'mean')
).reset_index()

customer_summary_clean[['frequency', 'total_spend', 'avg_spend', 'avg_rating']].corr()

,frequency,total_spend,avg_spend,avg_rating
frequency,1.000000,0.890245,-0.056133,-0.073586
total_spend,0.890245,1.000000,0.392299,-0.114278
avg_spend,-0.056133,0.392299,1.000000,-0.121310
avg_rating,-0.073586,-0.114278,-0.121310,1.000000


In [53]:
customer_summary_clean = df_clean.groupby('Customer Reference ID').agg(
    frequency=('Item Purchased', 'count'),
    total_spend=('Purchase Amount (USD)', 'sum'),
    avg_spend=('Purchase Amount (USD)', 'mean'),
    avg_rating=('Review Rating', 'mean'),
    last_purchase=('Date Purchase', 'max')
).reset_index()

customer_summary_clean['last_purchase'] = pd.to_datetime(customer_summary_clean['last_purchase'], format='%d-%m-%Y')
most_recent_date = customer_summary_clean['last_purchase'].max()
customer_summary_clean['days_since_last_purchase'] = (most_recent_date - customer_summary_clean['last_purchase']).dt.days

customer_summary_clean[['frequency', 'total_spend', 'avg_rating', 'days_since_last_purchase']].corr()

,frequency,total_spend,avg_rating,days_since_last_purchase
frequency,1.000000,0.890245,-0.073586,0.00113
total_spend,0.890245,1.000000,-0.114278,0.01824
avg_rating,-0.073586,-0.114278,1.000000,0.04814
days_since_last_purchase,0.001130,0.018240,0.048140,1.00000


In [55]:
#Conclusion: Customer profitability in this segment is driven primarily by purchase frequency, not average transaction value (r=0.89) 
#after excluding 43 outlier transactions that initially created a misleading picture; r=0.90 for average value). 
#Customer satisfaction (ratings) shows no measurable link to either profitability or customer activity/churn risk (r ≈ 0 in all tests)
#satisfaction should therefore be measured and managed as an independent KPI, rather than assumed to reflect or predict profitability.


In [61]:
#Dataset 
#Fashion Retail Sales Dataset
#https://www.kaggle.com/datasets/atharvasoundankar/fashion-retail-sales